[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vkoul/advanced-ab-testing/blob/main/assignments/week01_assignment.ipynb)

> **Run this notebook interactively:** Click the badge above to open in Google Colab.

# Week 1 Assignment: Statistical Foundations

## QuickCart — Validating the Testing Pipeline

You've just joined QuickCart's experimentation platform team. Before running any real A/B tests, your first task is to **build and validate the core statistical testing infrastructure**.

The team needs confidence that the tests produce correct error rates before trusting them with business decisions worth millions in revenue.

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt

---

## Task 1: Confidence Interval for a Bernoulli Parameter

### Context

QuickCart tracks **conversion rate** (proportion of visitors who make a purchase). Before comparing groups, you need to know how to build a confidence interval for a single proportion.

### Problem

Implement a function that computes a **95% confidence interval** for the parameter $p$ of a Bernoulli distribution, given a sample of 0s and 1s.

Recall the formula for the CI of a proportion:

$$\hat{p} \pm z_{\alpha/2} \cdot \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

where $z_{0.025} = 1.96$ for a 95% CI.

**Requirements:**
- Return a tuple `(left_bound, right_bound)`
- Clip bounds to [0, 1] (a probability can't be negative or exceed 1)

<details>
<summary>Hint (click to expand)</summary>

1. Compute $\hat{p}$ as the sample mean
2. Compute the standard error: $\sqrt{\hat{p}(1-\hat{p})/n}$
3. Use `np.clip` to constrain the bounds

</details>

In [ ]:
def get_bernoulli_confidence_interval(values: np.ndarray) -> tuple:
    """Compute 95% confidence interval for the Bernoulli parameter.
    
    Parameters
    ----------
    values : np.ndarray
        Array of 0s and 1s (binary outcomes).
    
    Returns
    -------
    tuple
        (left_bound, right_bound) of the 95% confidence interval,
        clipped to [0, 1].
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test your implementation
np.random.seed(42)
test_values = np.random.randint(2, size=1000)
ci = get_bernoulli_confidence_interval(test_values)
print(f"Sample proportion: {test_values.mean():.4f}")
print(f"95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")

# Verify
assert ci[0] >= 0 and ci[1] <= 1, "Bounds must be in [0, 1]"
assert ci[0] < test_values.mean() < ci[1], "Mean should be inside CI"
assert 0.02 < (ci[1] - ci[0]) < 0.10, "CI width seems off"
print("All checks passed!")

---

## Task 2: Estimating Type I Error Rate

### Context

Your team has historical data from a previous experiment. Before reusing this data split for future tests, you want to verify that comparing the pilot and control groups with a t-test produces a **calibrated Type I error rate** (~0.05).

The idea: bootstrap resample from both groups (treating them as coming from the same population) and check how often the t-test incorrectly rejects the null.

### Problem

Implement a function that:
1. Takes pilot and control DataFrames with a metric column
2. Runs `n_iter` bootstrap iterations:
   - Resample (with replacement) from the pilot group (same size as original)
   - Resample (with replacement) from the control group (same size as original)
   - Run `ttest_ind` on the two resamples
3. Returns the fraction of iterations where p-value < alpha

**Why this works**: If both groups come from the same distribution, resampling from them independently should produce significant differences only ~alpha of the time.

<details>
<summary>Hint (click to expand)</summary>

1. Extract values as numpy arrays from both DataFrames
2. In each iteration, use `np.random.choice(values, size=len_original, replace=True)`
3. Compare each p-value to alpha and take the mean of the boolean array

</details>

In [ ]:
def estimate_first_type_error(
    df_pilot_group: pd.DataFrame,
    df_control_group: pd.DataFrame,
    metric_name: str,
    alpha: float = 0.05,
    n_iter: int = 10000,
    seed: int = None
) -> float:
    """Estimate Type I error rate via bootstrap resampling.
    
    Bootstrap resamples from pilot and control groups of the same sizes,
    runs t-test on each pair, and returns the fraction of significant results.
    
    Parameters
    ----------
    df_pilot_group : pd.DataFrame
        DataFrame with pilot group data.
    df_control_group : pd.DataFrame
        DataFrame with control group data.
    metric_name : str
        Name of the column containing the metric.
    alpha : float
        Significance level (default 0.05).
    n_iter : int
        Number of bootstrap iterations (default 10000).
    seed : int, optional
        Random seed for reproducibility.
    
    Returns
    -------
    float
        Estimated Type I error rate.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test your implementation
# When both groups come from the same distribution, Type I error should be ~alpha
np.random.seed(0)
n = 200
metric = 'revenue'

df_pilot = pd.DataFrame({metric: np.random.normal(50, 10, n)})
df_control = pd.DataFrame({metric: np.random.normal(50, 10, n)})

error_rate = estimate_first_type_error(df_pilot, df_control, metric, n_iter=5000, seed=42)
print(f"Estimated Type I error rate: {error_rate:.4f}")
print(f"Expected: ~0.05")

# Should be reasonably close to 0.05
assert 0.02 < error_rate < 0.10, f"Type I error rate {error_rate:.3f} seems miscalibrated"
print("Check passed!")

---

## Task 3: Estimating Type II Error Rate

### Context

Now you want to understand the **power** of your test. Given your sample sizes, what's the probability of *missing* a real effect? And how does this change with different effect sizes?

### Problem

Implement a function that:
1. Takes pilot and control DataFrames and a list of **effect sizes** (e.g., `[1.03, 1.05, 1.10]` meaning 3%, 5%, 10% relative increases)
2. For each effect size, runs `n_iter` bootstrap iterations:
   - Resample from the pilot group
   - **Add a synthetic effect**: shift the resampled pilot values by adding `N(mean * (effect - 1), std / 10)` noise
   - Resample from the control group
   - Run `ttest_ind`
3. Returns a dictionary `{effect_size: type_II_error_rate}` where Type II error = fraction of iterations where p-value >= alpha (failed to detect the effect)

<details>
<summary>Hint (click to expand)</summary>

1. For each effect, the shift amount is: `np.random.normal(mean_pilot * (effect - 1), std_pilot / 10, len_pilot)`
2. Add this shift to the bootstrapped pilot values before running the t-test
3. Type II error = `np.mean(p_values >= alpha)` (didn't reject when we should have)

</details>

In [ ]:
def estimate_second_type_error(
    df_pilot_group: pd.DataFrame,
    df_control_group: pd.DataFrame,
    metric_name: str,
    effects: list,
    alpha: float = 0.05,
    n_iter: int = 10000,
    seed: int = None
) -> dict:
    """Estimate Type II error rate for different effect sizes.
    
    For each effect size, bootstraps pilot and control groups, adds a
    synthetic effect to the pilot, and measures how often the t-test
    fails to detect the difference.
    
    Parameters
    ----------
    df_pilot_group : pd.DataFrame
        DataFrame with pilot group data.
    df_control_group : pd.DataFrame
        DataFrame with control group data.
    metric_name : str
        Name of the column containing the metric.
    effects : list of float
        Effect sizes as multiplicative factors.
        1.03 means a 3% increase, 0.97 means a 3% decrease.
    alpha : float
        Significance level (default 0.05).
    n_iter : int
        Number of bootstrap iterations (default 10000).
    seed : int, optional
        Random seed for reproducibility.
    
    Returns
    -------
    dict
        {effect_size: type_II_error_rate}
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test your implementation
np.random.seed(0)
n = 200
metric = 'revenue'

df_pilot = pd.DataFrame({metric: np.random.normal(50, 10, n)})
df_control = pd.DataFrame({metric: np.random.normal(50, 10, n)})

effects = [1.03, 1.05, 1.10, 1.20]
errors = estimate_second_type_error(
    df_pilot, df_control, metric,
    effects=effects, n_iter=5000, seed=42
)

print("Type II error rates by effect size:")
print(f"{'Effect':>10} {'Type II Error':>15} {'Power (1-beta)':>15}")
print("-" * 42)
for effect, error in errors.items():
    print(f"{effect:>10.0%} {error:>15.3f} {1-error:>15.3f}")

# Larger effects should be easier to detect (lower Type II error)
error_values = list(errors.values())
assert error_values == sorted(error_values, reverse=True), \
    "Larger effects should have lower Type II error"
print("\nCheck passed! Larger effects are easier to detect.")

In [ ]:
# Visualize the power curve
plt.figure(figsize=(10, 6))
effect_pcts = [(e - 1) * 100 for e in errors.keys()]
power_values = [1 - e for e in errors.values()]

plt.plot(effect_pcts, power_values, 'bo-', markersize=8)
plt.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='Desired power (0.80)')
plt.xlabel('Effect size (% relative lift)')
plt.ylabel('Power (1 - Type II error)')
plt.title('Power Curve: Probability of Detecting an Effect')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim([0, 1.05])
plt.show()

---

## Task 4 (Bonus): Compare Test Robustness

### Context

QuickCart's revenue data has occasional extreme outliers (bulk corporate orders). You want to recommend which test to use as the default in the experimentation platform.

### Problem

Design a simulation that:
1. Generates control data from N(0, 1) with 1% of values replaced by outliers drawn from N(0, 100)
2. Generates treatment data from N(0.2, 1) (a real effect exists)
3. Compares detection rates (power) of t-test vs Mann-Whitney across different outlier fractions (0%, 1%, 5%, 10%)

Write a brief recommendation (2-3 sentences) for which test to use as default at QuickCart.

In [ ]:
# YOUR CODE HERE
# Design and run the simulation, then plot the results

**Your recommendation:**

*Write your recommendation here.*